# Rayhana Basil Disease Detection with YOLOv8

Rayhana is a basil-only smart care app. This notebook trains a basil object detector for three labels: fusarium, healthy, and powdery.

The repository is code-only. The dataset is downloaded from Roboflow inside Colab, prepared into a fresh YOLO split, inspected, trained, evaluated, and only then considered for backend integration.

## 1. Run Context

The latest YOLOv8n GPU run is still weak: mAP50 = 0.2206, mAP50-95 = 0.1224, precision = 0.1833, and recall = 0.4482. The decision remains: Not ready for backend integration.

That result is useful, though. It tells me the pipeline works, but the current baseline does not learn the basil disease signal well enough. With only 305 basil images, YOLOv8n is probably too small and too brittle for this label set. The next controlled experiment is YOLOv8s with moderate, safe augmentation and persistent Google Drive outputs.

## 2. Environment Setup

Use Runtime > Change runtime type > GPU before running the training section. The notebook checks this early and checks again immediately before training.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/Mohamed-515/rayhana-ai.git'
PROJECT_DIR = Path('/content/rayhana-ai')

In [ ]:
try:
    gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
except FileNotFoundError:
    gpu_check = subprocess.CompletedProcess(['nvidia-smi'], returncode=1, stdout='', stderr='nvidia-smi not found')

GPU_AVAILABLE = gpu_check.returncode == 0

if GPU_AVAILABLE:
    print(gpu_check.stdout.splitlines()[0])
    TRAINING_DEVICE = 0
else:
    print('No GPU detected. Dataset exploration can continue, but training will stop before model.fit/train.')
    print('In Colab: Runtime > Change runtime type > Hardware accelerator > GPU')
    TRAINING_DEVICE = None

In [ ]:
if not (PROJECT_DIR / 'src').exists():
    !git clone -q {REPO_URL} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print('Project:', PROJECT_DIR)

In [ ]:
!pip -q install -r requirements.txt roboflow

In [ ]:
from collections import Counter
import json
import random
import shutil

import matplotlib.pyplot as plt
import pandas as pd
import yaml
import torch
from IPython.display import Markdown, display
from PIL import Image, ImageDraw, ImageFont, UnidentifiedImageError
from ultralytics import YOLO

In [ ]:
TORCH_CUDA_AVAILABLE = torch.cuda.is_available()
GPU_AVAILABLE = GPU_AVAILABLE and TORCH_CUDA_AVAILABLE

if GPU_AVAILABLE:
    print('Torch CUDA device:', torch.cuda.get_device_name(0))
    TRAINING_DEVICE = 0
else:
    print('Torch cannot see a CUDA device. Exploration is fine, but training remains blocked.')
    TRAINING_DEVICE = None

## 3. Google Drive Output Safety

Colab runtimes can disconnect without warning. All training runs, exported weights, metrics, and prediction images are written to Google Drive so the experiment survives a runtime reset.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/rayhana-ai-outputs')
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Drive output root:', DRIVE_OUTPUT_ROOT)

In [ ]:
DATASET_DIR = PROJECT_DIR / 'data/processed/basil_yolo_detection'
DATA_YAML = DATASET_DIR / 'data.yaml'
RAW_ROBOFLOW_DIR = PROJECT_DIR / 'data/raw/basil_roboflow_yolov8'

LOCAL_REPORTS_DIR = PROJECT_DIR / 'reports'
REPORTS_DIR = DRIVE_OUTPUT_ROOT / 'reports'
METRICS_DIR = DRIVE_OUTPUT_ROOT / 'metrics'
FIGURES_DIR = DRIVE_OUTPUT_ROOT / 'figures'
MODEL_EXPORT_DIR = DRIVE_OUTPUT_ROOT / 'models'
DATASET_REPORT_DIR = DRIVE_OUTPUT_ROOT / 'dataset_reports'
YOLO_PROJECT = str(DRIVE_OUTPUT_ROOT / 'yolo_runs')
RUN_NAME = 'basil_yolov8n_baseline'

for output_dir in [REPORTS_DIR, METRICS_DIR, FIGURES_DIR, MODEL_EXPORT_DIR, DATASET_REPORT_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)

print('Raw dataset target:', RAW_ROBOFLOW_DIR)
print('Processed dataset target:', DATASET_DIR)
print('YOLO runs will be saved to:', YOLO_PROJECT)

## 4. Download Dataset from Roboflow

Download the Basil YOLOv8 export inside Colab. Use either the pasted-code cell or the API-key cell. Do not commit downloaded images to GitHub.

In [ ]:
# Option A: paste Roboflow's generated YOLOv8 download code below.
# It should end with something like: dataset = version.download('yolov8')
# Leave this cell unchanged if you prefer the API-key cell.

ROBOFLOW_DATASET_DIR = None

# Paste Roboflow code under this line.
# from roboflow import Roboflow
# rf = Roboflow(api_key='PASTE_API_KEY')
# project = rf.workspace('WORKSPACE').project('PROJECT')
# version = project.version(1)
# dataset = version.download('yolov8')

if 'dataset' in globals() and hasattr(dataset, 'location'):
    ROBOFLOW_DATASET_DIR = Path(dataset.location)
    print('Roboflow dataset:', ROBOFLOW_DATASET_DIR)
else:
    print('Paste Roboflow code here, or use the API-key cell below.')

In [ ]:
# Option B: fill these values, or store the key in Colab Secrets as ROBOFLOW_API_KEY.
ROBOFLOW_API_KEY = ''
ROBOFLOW_WORKSPACE = ''
ROBOFLOW_PROJECT = ''
ROBOFLOW_VERSION = ''

In [ ]:
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = ROBOFLOW_API_KEY or (userdata.get('ROBOFLOW_API_KEY') or '')
except Exception:
    pass

if ROBOFLOW_API_KEY and ROBOFLOW_WORKSPACE and ROBOFLOW_PROJECT and ROBOFLOW_VERSION:
    from roboflow import Roboflow

    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    dataset = project.version(int(ROBOFLOW_VERSION)).download('yolov8')
    ROBOFLOW_DATASET_DIR = Path(dataset.location)
    print('Roboflow dataset:', ROBOFLOW_DATASET_DIR)
else:
    print('Skipped API-key download. Fill the four values above if you are not using pasted Roboflow code.')

In [ ]:
if ROBOFLOW_DATASET_DIR is not None:
    ROBOFLOW_DATASET_DIR = Path(ROBOFLOW_DATASET_DIR)

if ROBOFLOW_DATASET_DIR is None:
    candidates = []
    for data_yaml in Path('/content').rglob('data.yaml'):
        candidate = data_yaml.parent
        has_yolo_split = any((candidate / split / 'images').exists() for split in ['train', 'valid', 'test'])
        if has_yolo_split:
            candidates.append(candidate)
    candidates = sorted(candidates, key=lambda item: item.stat().st_mtime, reverse=True)
    ROBOFLOW_DATASET_DIR = candidates[0] if candidates else None

assert ROBOFLOW_DATASET_DIR is not None, 'Download the Roboflow YOLOv8 dataset first.'
assert ROBOFLOW_DATASET_DIR.exists(), f'Missing Roboflow folder: {ROBOFLOW_DATASET_DIR}'
print('Using Roboflow folder:', ROBOFLOW_DATASET_DIR)

## 5. Prepare Train/Valid/Test Split

Roboflow exports may arrive with one split or with existing train/valid/test folders. This staging step collects all available YOLO image-label pairs into Rayhana's raw dataset folder. The project script then creates the reproducible 70/20/10 processed split used for training.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
SPLITS = ['train', 'valid', 'test']

In [ ]:
def roboflow_split_dirs(root):
    split_dirs = []
    for split in SPLITS:
        image_dir = root / split / 'images'
        label_dir = root / split / 'labels'
        if image_dir.exists() and label_dir.exists():
            split_dirs.append((split, image_dir, label_dir))

    image_dir = root / 'images'
    label_dir = root / 'labels'
    if image_dir.exists() and label_dir.exists():
        split_dirs.append(('root', image_dir, label_dir))

    return split_dirs

In [ ]:
def stage_roboflow_for_rayhana(source_root, raw_root):
    split_dirs = roboflow_split_dirs(source_root)
    assert split_dirs, f'No YOLO image/label folders found under {source_root}'

    if raw_root.exists():
        shutil.rmtree(raw_root)

    target_images = raw_root / 'train' / 'images'
    target_labels = raw_root / 'train' / 'labels'
    target_images.mkdir(parents=True, exist_ok=True)
    target_labels.mkdir(parents=True, exist_ok=True)

    staged = []
    missing_labels = []

    for split, image_dir, label_dir in split_dirs:
        for image_path in sorted(image_dir.iterdir()):
            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            label_path = label_dir / f'{image_path.stem}.txt'
            if not label_path.exists():
                missing_labels.append((split, image_path.name))
                continue

            target_stem = image_path.stem
            target_image_path = target_images / f'{target_stem}{image_path.suffix}'
            target_label_path = target_labels / f'{target_stem}.txt'

            if target_image_path.exists() or target_label_path.exists():
                target_stem = f'{split}_{image_path.stem}'
                target_image_path = target_images / f'{target_stem}{image_path.suffix}'
                target_label_path = target_labels / f'{target_stem}.txt'

            shutil.copy2(image_path, target_image_path)
            shutil.copy2(label_path, target_label_path)
            staged.append({'source_split': split, 'image': target_image_path.name})

    return staged, missing_labels

In [ ]:
staged_pairs, missing_raw_labels = stage_roboflow_for_rayhana(ROBOFLOW_DATASET_DIR, RAW_ROBOFLOW_DIR)

print(f'Staged image/label pairs: {len(staged_pairs)}')
print(f'Missing labels skipped: {len(missing_raw_labels)}')
assert staged_pairs, 'No labeled images were staged.'

In [ ]:
!python -m src.data.prepare_basil_yolo_split

assert DATA_YAML.exists(), f'Missing processed data.yaml: {DATA_YAML}'
print('Processed dataset ready:', DATASET_DIR)

In [ ]:
split_report_path = PROJECT_DIR / 'reports/basil_yolo_split_report.json'
split_report = json.loads(split_report_path.read_text(encoding='utf-8'))

DRIVE_SPLIT_REPORT_PATH = DATASET_REPORT_DIR / 'basil_yolo_split_report.json'
shutil.copy2(split_report_path, DRIVE_SPLIT_REPORT_PATH)

print('Drive split report:', DRIVE_SPLIT_REPORT_PATH)
split_report

## 6. Dataset Exploration

Before training, I want a compact picture of the data: how many images exist, whether the split is reasonable, how labels are distributed, and whether bounding boxes look like leaf-level disease annotations rather than accidental full-image boxes.

In [ ]:
for split in SPLITS:
    assert (DATASET_DIR / split / 'images').exists(), f'Missing {split}/images'
    assert (DATASET_DIR / split / 'labels').exists(), f'Missing {split}/labels'

assert DATA_YAML.exists(), 'Missing data.yaml'
print('Dataset folders are ready.')

In [ ]:
with DATA_YAML.open('r', encoding='utf-8') as f:
    data_config = yaml.safe_load(f)

data_config

In [ ]:
class_names = data_config['names']
id_to_class = {idx: name for idx, name in enumerate(class_names)}
class_to_id = {name: idx for idx, name in id_to_class.items()}

class_names

In [ ]:
def image_paths_for_split(split):
    image_dir = DATASET_DIR / split / 'images'
    return sorted(path for path in image_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)


def label_paths_for_split(split):
    label_dir = DATASET_DIR / split / 'labels'
    return sorted(label_dir.glob('*.txt'))


def label_path_for_image(split, image_path):
    return DATASET_DIR / split / 'labels' / f'{image_path.stem}.txt'

In [ ]:
split_images = {split: image_paths_for_split(split) for split in SPLITS}
split_labels = {split: label_paths_for_split(split) for split in SPLITS}

image_counts = {split: len(paths) for split, paths in split_images.items()}
label_counts = {split: len(paths) for split, paths in split_labels.items()}

total_images = sum(image_counts.values())
print(f'Total images: {total_images}')

In [ ]:
def read_yolo_label(label_path):
    rows = []
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        try:
            rows.append({
                'class_id': int(parts[0]),
                'x_center': float(parts[1]),
                'y_center': float(parts[2]),
                'width': float(parts[3]),
                'height': float(parts[4]),
            })
        except ValueError:
            continue
    return rows

In [ ]:
annotation_records = []
box_records = []

for split, image_paths in split_images.items():
    for image_path in image_paths:
        label_path = label_path_for_image(split, image_path)
        rows = read_yolo_label(label_path) if label_path.exists() else []
        annotation_records.append({
            'split': split,
            'image': image_path.name,
            'box_count': len(rows),
        })
        for row in rows:
            class_name = id_to_class.get(row['class_id'], f'unknown_{row["class_id"]}')
            box_records.append({
                'split': split,
                'image': image_path.name,
                'class_id': row['class_id'],
                'class_name': class_name,
                'x_center': row['x_center'],
                'y_center': row['y_center'],
                'width': row['width'],
                'height': row['height'],
                'area': row['width'] * row['height'],
            })

image_df = pd.DataFrame(annotation_records)
box_df = pd.DataFrame(box_records)

image_df.head()

In [ ]:
split_summary = image_df.groupby('split').agg(
    images=('image', 'count'),
    annotations=('box_count', 'sum'),
    avg_boxes_per_image=('box_count', 'mean'),
).reindex(SPLITS)

split_summary['labels'] = pd.Series(label_counts)
split_summary['image_pct'] = split_summary['images'] / split_summary['images'].sum()
split_summary.round(3)

In [ ]:
display(Markdown(
    f'**Dataset size:** {total_images} images and {len(box_df)} labeled boxes. '
    f'Average boxes per image: {image_df["box_count"].mean():.2f}.'
))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(split_summary.index, split_summary['images'], color=['#3B82F6', '#10B981', '#F59E0B'])
ax.set_title('Images per Split')
ax.set_xlabel('Split')
ax.set_ylabel('Image Count')
for idx, value in enumerate(split_summary['images']):
    ax.text(idx, value, str(int(value)), ha='center', va='bottom')
plt.tight_layout()
plt.show()

Observation: this split chart is the first sanity check. If validation or test has only a small number of images, metrics will jump around between runs. For Rayhana, that means a low score is meaningful, but a single high score still needs careful visual review.

In [ ]:
class_distribution = pd.crosstab(box_df['split'], box_df['class_name']).reindex(index=SPLITS, columns=class_names, fill_value=0)
class_distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
class_distribution.plot(kind='bar', ax=ax)
ax.set_title('Annotation Counts by Class')
ax.set_xlabel('Split')
ax.set_ylabel('Box Count')
ax.legend(title='Class')
plt.tight_layout()
plt.show()

Observation: class balance matters more than total image count here. A basil detector that sees many more healthy leaves than disease examples can look calm in demos while missing the exact cases Rayhana needs to catch.

In [ ]:
box_density = image_df['box_count'].describe().to_frame(name='boxes_per_image')
box_density.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
image_df['box_count'].plot(kind='hist', bins=range(0, int(image_df['box_count'].max()) + 2), ax=ax)
ax.set_title('Boxes per Image')
ax.set_xlabel('Number of Boxes')
ax.set_ylabel('Image Count')
plt.tight_layout()
plt.show()

Observation: boxes-per-image tells me whether the task is usually single-object or crowded. If most images have one box, missed detections are easy to spot; if many images have several boxes, recall becomes more important than a clean-looking single prediction.

In [ ]:
box_size_summary = box_df.groupby('class_name')[['width', 'height', 'area']].describe().round(3)
box_size_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for class_name in class_names:
    class_boxes = box_df[box_df['class_name'] == class_name]
    ax.hist(class_boxes['area'], bins=20, alpha=0.55, label=class_name)
ax.set_title('Bounding Box Area Distribution')
ax.set_xlabel('Normalized Box Area')
ax.set_ylabel('Box Count')
ax.legend()
plt.tight_layout()
plt.show()

Observation: very tiny boxes are hard for YOLO to learn from, and very large boxes may mean the label is describing the whole leaf instead of the disease region. Either pattern is not automatically wrong, but it changes how I read weak recall.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for class_name in class_names:
    class_boxes = box_df[box_df['class_name'] == class_name]
    ax.scatter(class_boxes['width'], class_boxes['height'], s=18, alpha=0.65, label=class_name)
ax.set_title('Bounding Box Width vs Height')
ax.set_xlabel('Normalized Width')
ax.set_ylabel('Normalized Height')
ax.legend()
plt.tight_layout()
plt.show()

Observation: this scatterplot is a label-style check. A tight cluster can be healthy; it can also reveal that all annotations were drawn with the same habit. Outliers are worth opening manually before trusting training curves.

In [ ]:
class_totals = box_df['class_name'].value_counts().reindex(class_names, fill_value=0)
imbalance_df = pd.DataFrame({
    'annotations': class_totals,
    'share': class_totals / class_totals.sum(),
})
imbalance_df['relative_to_largest'] = imbalance_df['annotations'] / imbalance_df['annotations'].max()
imbalance_df.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
imbalance_df['share'].plot(kind='bar', ax=ax, color='#6366F1')
ax.set_title('Class Share Across All Annotations')
ax.set_xlabel('Class')
ax.set_ylabel('Share of Boxes')
ax.set_ylim(0, max(0.5, imbalance_df['share'].max() + 0.1))
for idx, value in enumerate(imbalance_df['share']):
    ax.text(idx, value, f'{value:.1%}', ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
largest_class = imbalance_df['annotations'].idxmax()
smallest_class = imbalance_df['annotations'].idxmin()
imbalance_ratio = imbalance_df.loc[largest_class, 'annotations'] / max(1, imbalance_df.loc[smallest_class, 'annotations'])

print(f'Largest class: {largest_class}')
print(f'Smallest class: {smallest_class}')
print(f'Largest/smallest annotation ratio: {imbalance_ratio:.2f}x')

Observation: if one disease class is much smaller than the others, a weak class-level result is expected. The next dataset pass should prioritize the rare class instead of only adding more basil images in general.

## 7. Data Validation

These checks are intentionally direct. I want the notebook to fail before training if labels are malformed, images are unreadable, or image-label pairs do not match.

In [ ]:
label_format_issues = []

for split, label_paths in split_labels.items():
    for label_path in label_paths:
        for line_number, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), start=1):
            parts = line.strip().split()
            if len(parts) != 5:
                label_format_issues.append((split, label_path.name, line_number, 'wrong field count'))
                continue
            try:
                int(parts[0])
                [float(value) for value in parts[1:]]
            except ValueError:
                label_format_issues.append((split, label_path.name, line_number, 'non-numeric value'))

print(f'Label format issues: {len(label_format_issues)}')
label_format_issues[:5]

In [ ]:
corrupted_images = []

for split, image_paths in split_images.items():
    for image_path in image_paths:
        try:
            with Image.open(image_path) as image:
                image.verify()
        except (UnidentifiedImageError, OSError) as error:
            corrupted_images.append({'split': split, 'path': str(image_path), 'error': str(error)})

print(f'Corrupted images: {len(corrupted_images)}')

In [ ]:
missing_labels = []
missing_images = []

for split in SPLITS:
    image_stems = {path.stem for path in split_images[split]}
    label_stems = {path.stem for path in split_labels[split]}
    missing_labels.extend((split, stem) for stem in sorted(image_stems - label_stems))
    missing_images.extend((split, stem) for stem in sorted(label_stems - image_stems))

print(f'Missing labels: {len(missing_labels)}')
print(f'Missing images: {len(missing_images)}')

In [ ]:
invalid_class_rows = []
valid_class_ids = set(id_to_class.keys())

for split, label_paths in split_labels.items():
    for label_path in label_paths:
        for line_number, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), start=1):
            parts = line.strip().split()
            if not parts:
                continue
            try:
                class_id = int(parts[0])
            except ValueError:
                invalid_class_rows.append((split, label_path.name, line_number, parts[0]))
                continue
            if class_id not in valid_class_ids:
                invalid_class_rows.append((split, label_path.name, line_number, class_id))

print(f'Invalid class IDs: {len(invalid_class_rows)}')

In [ ]:
validation_summary = {
    'corrupted_images': len(corrupted_images),
    'missing_labels': len(missing_labels),
    'missing_images': len(missing_images),
    'invalid_class_ids': len(invalid_class_rows),
    'label_format_issues': len(label_format_issues),
}

validation_summary

In [ ]:
assert all(count == 0 for count in validation_summary.values()), validation_summary
print('Validation passed. The dataset is structurally ready for training.')

## 8. Visual Inspection

Numbers can miss label problems. I first look at the raw images without boxes, then the same images with boxes and class labels. This keeps the review grounded in actual basil leaves rather than only charts.

In [ ]:
random.seed(42)
all_image_records = [(split, path) for split, paths in split_images.items() for path in paths]
inspection_records = random.sample(all_image_records, k=min(6, len(all_image_records)))
inspection_records[:2]

In [ ]:
def show_plain_images(records, title):
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    axes = axes.flatten()

    for ax, (split, image_path) in zip(axes, records):
        image = Image.open(image_path).convert('RGB')
        ax.imshow(image)
        ax.set_title(f'{split}: {image_path.name[:28]}')
        ax.axis('off')

    for ax in axes[len(records):]:
        ax.axis('off')

    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
show_plain_images(inspection_records, 'Random Basil Images Without Boxes')

Observation: before drawing boxes, I am checking whether the images actually look like Rayhana's target domain: basil leaves, phone-like framing, and symptoms that a user might scan. If the raw image mix is off, better training will only make a better model for the wrong setting.

In [ ]:
def yolo_to_pixels(row, image_width, image_height):
    x_center = row['x_center'] * image_width
    y_center = row['y_center'] * image_height
    box_width = row['width'] * image_width
    box_height = row['height'] * image_height
    left = max(0, x_center - box_width / 2)
    top = max(0, y_center - box_height / 2)
    right = min(image_width, x_center + box_width / 2)
    bottom = min(image_height, y_center + box_height / 2)
    return left, top, right, bottom

In [ ]:
def draw_annotations(image_path, label_path):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    width, height = image.size

    for row in read_yolo_label(label_path):
        if row['class_id'] not in id_to_class:
            continue
        class_name = id_to_class[row['class_id']]
        left, top, right, bottom = yolo_to_pixels(row, width, height)
        line_width = max(2, width // 250)
        draw.rectangle((left, top, right, bottom), outline='red', width=line_width)
        draw.text((left + 3, max(0, top - 12)), class_name, fill='red', font=font)

    return image

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()

for ax, (split, image_path) in zip(axes, inspection_records):
    label_path = label_path_for_image(split, image_path)
    annotated = draw_annotations(image_path, label_path)
    ax.imshow(annotated)
    ax.set_title(f'{split}: {image_path.name[:28]}')
    ax.axis('off')

for ax in axes[len(inspection_records):]:
    ax.axis('off')

fig.suptitle('Same Images With YOLO Boxes and Class Labels', y=1.02)
plt.tight_layout()
plt.show()

Observation: I want boxes that land on basil leaf regions with visible class evidence. If a box covers the full image, misses the symptom, or labels an ambiguous area too confidently, that sample should be reviewed before spending time on larger models.

In [ ]:
random.seed(7)
extra_annotation_records = random.sample(all_image_records, k=min(6, len(all_image_records)))
show_plain_images(extra_annotation_records, 'Second Random Image Check')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
axes = axes.flatten()

for ax, (split, image_path) in zip(axes, extra_annotation_records):
    label_path = label_path_for_image(split, image_path)
    annotated = draw_annotations(image_path, label_path)
    ax.imshow(annotated)
    ax.set_title(f'{split}: {image_path.name[:28]}')
    ax.axis('off')

for ax in axes[len(extra_annotation_records):]:
    ax.axis('off')

fig.suptitle('Second Random Check With Boxes', y=1.02)
plt.tight_layout()
plt.show()

Observation: a second random sample helps avoid designing conclusions around one lucky grid. I am looking for repeated patterns: tiny boxes, class confusion, inconsistent leaf framing, or symptoms that are visually too subtle for the current label set.

## 9. Training Plan

The first experiment was YOLOv8n: a deliberately small baseline to test whether the pipeline and labels were learnable. The current GPU result, mAP50 = 0.2206, says the baseline is too weak for Rayhana.

The next experiment is not a random bigger-model jump. The basil dataset has only 305 images, so I want one controlled change: pretrained YOLOv8s plus stronger but still safe augmentation. The augmentation should help lighting, framing, and mild viewpoint variation without inventing unrealistic basil disease patterns.

In [ ]:
RUN_YOLOV8N_BASELINE = False

BASELINE_CONFIG = {
    'model': 'yolov8n.pt',
    'epochs': 120,
    'imgsz': 640,
    'batch': 8,
    'patience': 25,
    'optimizer': 'AdamW',
    'seed': 42,
    'project': YOLO_PROJECT,
    'name': RUN_NAME,
}

pd.DataFrame([BASELINE_CONFIG]).T.rename(columns={0: 'value'})

In [ ]:
def require_gpu_for_training():
    if not GPU_AVAILABLE:
        raise RuntimeError(
            'Training stopped: no Colab GPU is attached. '
            'Exploration can run on CPU, but Rayhana model training must use Runtime > Change runtime type > GPU.'
        )
    print('GPU safety check passed. Training will run on CUDA device 0.')

## 10. YOLOv8n Baseline Training

The YOLOv8n baseline is kept for reproducibility, but rerunning it is opt-in. The active next run is YOLOv8s because the GPU baseline is already below the integration threshold.

In [ ]:
if RUN_YOLOV8N_BASELINE:
    require_gpu_for_training()
    baseline_model = YOLO(BASELINE_CONFIG['model'])
    baseline_results = baseline_model.train(
        data=str(DATA_YAML),
        epochs=BASELINE_CONFIG['epochs'],
        imgsz=BASELINE_CONFIG['imgsz'],
        batch=BASELINE_CONFIG['batch'],
        patience=BASELINE_CONFIG['patience'],
        optimizer=BASELINE_CONFIG['optimizer'],
        seed=BASELINE_CONFIG['seed'],
        project=BASELINE_CONFIG['project'],
        name=BASELINE_CONFIG['name'],
        device=0,
        exist_ok=True,
    )
else:
    print('YOLOv8n baseline rerun skipped. Current GPU baseline: mAP50=0.2206, decision=Not ready for backend integration.')

## 11. Second Experiment: YOLOv8s + Safe Augmentation

This is the next controlled experiment for Rayhana. It uses pretrained yolov8s.pt, trains longer, waits longer before early stopping, and adds augmentation that matches plausible basil phone scans: mild rotations, modest scale changes, horizontal flips, lighting variation, mosaic, and a small amount of mixup.

I am not using aggressive vertical flips or copy-paste augmentation here. Those can create examples that are technically useful for generic detection but less trustworthy for a basil disease workflow.

In [ ]:
RUN_YOLOV8S_EXPERIMENT = True

In [ ]:
YOLOV8S_AUGMENTATION = {
    'hsv_h': 0.015,
    'hsv_s': 0.45,
    'hsv_v': 0.30,
    'degrees': 8.0,
    'translate': 0.08,
    'scale': 0.35,
    'shear': 2.0,
    'perspective': 0.0005,
    'fliplr': 0.5,
    'flipud': 0.0,
    'mosaic': 0.80,
    'mixup': 0.08,
    'copy_paste': 0.0,
    'close_mosaic': 20,
}

YOLOV8S_CONFIG = {
    'model': 'yolov8s.pt',
    'epochs': 150,
    'imgsz': 640,
    'batch': 8,
    'patience': 35,
    'optimizer': 'AdamW',
    'seed': 42,
    'project': YOLO_PROJECT,
    'name': 'basil_yolov8s_augmented',
    **YOLOV8S_AUGMENTATION,
}

pd.DataFrame([YOLOV8S_CONFIG]).T.rename(columns={0: 'value'})

In [ ]:
if RUN_YOLOV8S_EXPERIMENT:
    require_gpu_for_training()
    yolo8s_model = YOLO(YOLOV8S_CONFIG['model'])
    yolo8s_results = yolo8s_model.train(
        data=str(DATA_YAML),
        epochs=YOLOV8S_CONFIG['epochs'],
        imgsz=YOLOV8S_CONFIG['imgsz'],
        batch=YOLOV8S_CONFIG['batch'],
        patience=YOLOV8S_CONFIG['patience'],
        optimizer=YOLOV8S_CONFIG['optimizer'],
        seed=YOLOV8S_CONFIG['seed'],
        project=YOLOV8S_CONFIG['project'],
        name=YOLOV8S_CONFIG['name'],
        device=0,
        exist_ok=True,
        hsv_h=YOLOV8S_CONFIG['hsv_h'],
        hsv_s=YOLOV8S_CONFIG['hsv_s'],
        hsv_v=YOLOV8S_CONFIG['hsv_v'],
        degrees=YOLOV8S_CONFIG['degrees'],
        translate=YOLOV8S_CONFIG['translate'],
        scale=YOLOV8S_CONFIG['scale'],
        shear=YOLOV8S_CONFIG['shear'],
        perspective=YOLOV8S_CONFIG['perspective'],
        fliplr=YOLOV8S_CONFIG['fliplr'],
        flipud=YOLOV8S_CONFIG['flipud'],
        mosaic=YOLOV8S_CONFIG['mosaic'],
        mixup=YOLOV8S_CONFIG['mixup'],
        copy_paste=YOLOV8S_CONFIG['copy_paste'],
        close_mosaic=YOLOV8S_CONFIG['close_mosaic'],
    )
else:
    print('YOLOv8s experiment skipped. Set RUN_YOLOV8S_EXPERIMENT = True to run the next controlled experiment.')

## 12. Evaluation

After training, I evaluate the best checkpoint on the test split. The printed metrics are the minimum report needed before discussing backend integration.

In [ ]:
ACTIVE_EXPERIMENT = YOLOV8S_CONFIG if RUN_YOLOV8S_EXPERIMENT else BASELINE_CONFIG
ACTIVE_EXPERIMENT_NAME = ACTIVE_EXPERIMENT['name']
ACTIVE_MODEL_WEIGHTS = ACTIVE_EXPERIMENT['model']

print('Evaluation target:', ACTIVE_EXPERIMENT_NAME)
print('Outputs are in Google Drive:', DRIVE_OUTPUT_ROOT)

In [ ]:
RUN_DIR = Path(ACTIVE_EXPERIMENT['project']) / ACTIVE_EXPERIMENT_NAME
BEST_MODEL_PATH = RUN_DIR / 'weights' / 'best.pt'

assert BEST_MODEL_PATH.exists(), f'Missing trained model: {BEST_MODEL_PATH}'
best_model = YOLO(str(BEST_MODEL_PATH))
BEST_MODEL_PATH

In [ ]:
test_metrics = best_model.val(data=str(DATA_YAML), split='test', device=TRAINING_DEVICE)

In [ ]:
metrics_output = {
    'mAP50': float(test_metrics.box.map50),
    'mAP50_95': float(test_metrics.box.map),
    'precision': float(test_metrics.box.mp),
    'recall': float(test_metrics.box.mr),
}

for metric_name, metric_value in metrics_output.items():
    print(f'{metric_name}: {metric_value:.4f}')

Metric reading:

- mAP50: how well predicted boxes match labels at a forgiving overlap threshold. Good for a first object-detection check.
- mAP50-95: stricter average across overlap thresholds. This is harder and usually lower.
- Precision: when the model predicts a box, how often it is right.
- Recall: how many real annotated objects the model finds.

For Rayhana, recall matters because missing a visible basil issue is worse than showing a cautious low-confidence warning.

In [ ]:
map50 = metrics_output['mAP50']

if map50 < 0.50:
    quality_decision = 'Not ready for backend integration'
elif map50 < 0.75:
    quality_decision = 'Prototype only'
else:
    quality_decision = 'Good first integration candidate'

print(f'Result quality decision: {quality_decision}')

In [ ]:
VAL_DIR = Path(test_metrics.save_dir)
confusion_matrix_path = VAL_DIR / 'confusion_matrix.png'

if confusion_matrix_path.exists():
    image = Image.open(confusion_matrix_path)
    plt.figure(figsize=(7, 7))
    plt.imshow(image)
    plt.axis('off')
    plt.title('Confusion Matrix')
    plt.show()
else:
    print('Confusion matrix was not found in the validation output folder.')

In [ ]:
curve_files = ['results.png', 'P_curve.png', 'R_curve.png', 'PR_curve.png', 'F1_curve.png']

for file_name in curve_files:
    curve_path = RUN_DIR / file_name
    if curve_path.exists():
        image = Image.open(curve_path)
        plt.figure(figsize=(8, 5))
        plt.imshow(image)
        plt.axis('off')
        plt.title(file_name)
        plt.show()

Observation: I read the confusion matrix before celebrating a headline score. A model that works for healthy but misses fusarium or powdery is not useful enough for Rayhana, even if the aggregate metric improves.

## 13. Prediction Review

The goal here is not a pretty demo. I want to see both successful detections and failure cases, because those failures usually explain what the next dataset or training change should be.

In [ ]:
random.seed(42)
test_images = split_images['test']
prediction_images = random.sample(test_images, k=min(12, len(test_images)))

prediction_images[:3]

In [ ]:
PREDICTION_CONFIDENCE = 0.25

prediction_results = best_model.predict(
    source=[str(path) for path in prediction_images],
    conf=PREDICTION_CONFIDENCE,
    save=True,
    project=str(FIGURES_DIR),
    name=f'{ACTIVE_EXPERIMENT_NAME}_prediction_samples',
    device=TRAINING_DEVICE,
    exist_ok=True,
)

In [ ]:
prediction_rows = []

for result in prediction_results:
    image_name = Path(result.path).name
    if len(result.boxes) == 0:
        prediction_rows.append({
            'image': image_name,
            'class': None,
            'confidence': 0.0,
            'detections': 0,
        })
        continue

    for box in result.boxes:
        class_id = int(box.cls.item())
        prediction_rows.append({
            'image': image_name,
            'class': result.names[class_id],
            'confidence': float(box.conf.item()),
            'detections': len(result.boxes),
        })

prediction_df = pd.DataFrame(prediction_rows)
prediction_df.sort_values('confidence', ascending=False).head(20)

In [ ]:
PREDICTION_SAVE_DIR = Path(prediction_results[0].save_dir)
saved_predictions = sorted(path for path in PREDICTION_SAVE_DIR.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
saved_prediction_by_name = {path.name: path for path in saved_predictions}

len(saved_predictions)

In [ ]:
successful_images = []
for result in prediction_results:
    if len(result.boxes) > 0:
        saved_path = saved_prediction_by_name.get(Path(result.path).name)
        if saved_path is not None:
            successful_images.append(saved_path)

successful_images[:6]

In [ ]:
if successful_images:
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    axes = axes.flatten()
    for ax, image_path in zip(axes, successful_images[:6]):
        image = Image.open(image_path).convert('RGB')
        ax.imshow(image)
        ax.set_title(image_path.name[:35])
        ax.axis('off')
    for ax in axes[len(successful_images[:6]):]:
        ax.axis('off')
    fig.suptitle('Successful Detection Samples', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No successful detections at the current confidence threshold.')

In [ ]:
failed_images = [Path(result.path) for result in prediction_results if len(result.boxes) == 0]
failed_images[:6]

In [ ]:
if failed_images:
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))
    axes = axes.flatten()
    for ax, image_path in zip(axes, failed_images[:6]):
        image = Image.open(image_path).convert('RGB')
        ax.imshow(image)
        ax.set_title(f'No detection: {image_path.name[:28]}')
        ax.axis('off')
    for ax in axes[len(failed_images[:6]):]:
        ax.axis('off')
    fig.suptitle('No-Detection Cases', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No no-detection cases in this sample. Increase the sample size for a tougher review.')

In [ ]:
low_confidence_df = prediction_df[(prediction_df['detections'] > 0) & (prediction_df['confidence'] < 0.40)]
low_confidence_df.sort_values('confidence').head(20)

Observation: no-detection cases are likely false negatives when a visible annotated symptom exists. Low-confidence detections are also important: they may be correct but not stable enough for user-facing advice. These examples should drive the next labeling review before touching backend code.

## 14. Saving Outputs

The YOLO run directory is already in Google Drive. This section also exports the best weight, metrics JSON, and reviewed prediction images into stable Drive folders for handoff and comparison.

In [ ]:
FINAL_MODEL_DIR = MODEL_EXPORT_DIR
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_PATH = FINAL_MODEL_DIR / f'{ACTIVE_EXPERIMENT_NAME}_best.pt'
shutil.copy2(BEST_MODEL_PATH, FINAL_MODEL_PATH)

FINAL_MODEL_PATH

In [ ]:
metrics_output.update({
    'model': ACTIVE_MODEL_WEIGHTS.replace('.pt', ''),
    'experiment': ACTIVE_EXPERIMENT_NAME,
    'classes': class_names,
    'quality_decision': quality_decision,
    'best_model_path': str(FINAL_MODEL_PATH),
    'run_dir': str(RUN_DIR),
    'drive_output_root': str(DRIVE_OUTPUT_ROOT),
})

METRICS_PATH = METRICS_DIR / f'{ACTIVE_EXPERIMENT_NAME}_metrics.json'
METRICS_PATH.write_text(json.dumps(metrics_output, indent=2), encoding='utf-8')

METRICS_PATH

In [ ]:
FINAL_PREDICTION_DIR = FIGURES_DIR / f'{ACTIVE_EXPERIMENT_NAME}_predictions'
FINAL_PREDICTION_DIR.mkdir(parents=True, exist_ok=True)

for image_path in saved_predictions:
    shutil.copy2(image_path, FINAL_PREDICTION_DIR / image_path.name)

FINAL_PREDICTION_DIR

In [ ]:
print('Best model:', FINAL_MODEL_PATH)
print('Metrics:', METRICS_PATH)
print('Prediction samples:', FINAL_PREDICTION_DIR)
print('Training run:', RUN_DIR)
print('Decision:', quality_decision)

## 15. Developer Conclusion

The current YOLOv8n GPU result is weak: mAP50 = 0.2206, mAP50-95 = 0.1224, precision = 0.1833, and recall = 0.4482. The decision is still Not ready for backend integration.

The likely issue is not just training time. The basil dataset has only 305 images, and YOLOv8n is too weak for a reliable Rayhana detector on this label set. The next controlled experiment is YOLOv8s with pretrained yolov8s.pt, 150 epochs, patience 35, AdamW, and stronger but safe augmentation.

If YOLOv8s is still below mAP50 0.50, the right next move is dataset work: more basil images, cleaner class balance, and inspection of failed/no-detection examples. Backend integration should wait until metrics and visual prediction review are both acceptable.